In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from joblib import Parallel, delayed
from collections import Counter
from typing import List, Union, Sequence, Optional, Dict
from customkernels import Kernel1Full

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, hinge_loss, log_loss, brier_score_loss

import matplotlib.colors as mcolors

import matplotlib.lines as mlines
from sklearn.model_selection import KFold
from ucimlrepo import fetch_ucirepo 


In [33]:
def _bootstrap_indices(n: int, replace: bool) -> np.ndarray:
    """Return bootstrap sample indices of length *n*."""
    return np.random.choice(n, size=n, replace=replace)


def _select_feature_indices(p: int, m: int) -> np.ndarray:
    """Randomly select *m* distinct feature indices from *p* total."""
    return np.random.choice(p, size=m, replace=False)

In [ ]:
def subBagSVM_k1(X, y, m, cont_cols, no_svms=100, C_values=None, replace=True, n_jobs=1, random_state=None):

    rng = np.random.RandomState(random_state)
    n_samples, n_features = X.shape

    # Method fit one SVM with bootstrap + feature subspace
    def fit_one(seed):
        local_rng = np.random.RandomState(seed)
        idx_samples = local_rng.choice(n_samples, size=n_samples, replace=replace)
        feat_idx = local_rng.choice(n_features, size=m, replace=False)
        feat_names  = X.columns[feat_idx].tolist()
        X_sub = X.iloc[idx_samples, feat_idx]

        y_sub = y.iloc[idx_samples]

        cont_in_sub = [c for c in cont_cols if c in feat_names]


        gammas =  [0.5, 1.0, 2.0, 4.0] 
        alphas = [0.3, 0.5,  0.7, 1.0, 1.5]
        Cs = [0.1, 1, 10]

        best_accuracy = 0
        best_params = {}
        best_model = None

        # Add cross-validation (3-fold)
        kf = KFold(n_splits=3, shuffle=True, random_state=42)

        for γ in gammas:
            for α in alphas:
                for C in Cs:
                    accuracies = []
                    brier_scores = []

                    
                    # Cross-validation loop
                    for train_idx, val_idx in kf.split(X_sub):
                        X_tr, X_val = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
                        y_tr, y_val = y_sub.iloc[train_idx], y_sub.iloc[val_idx]
                        
                        # Train kernel and SVC
                        k1 = Kernel1Full(
                            continuous_vars=cont_in_sub,
                            alpha=α,
                            rbf_gamma=γ,
                            cat_gamma=γ
                        ).fit(X_tr, y_tr)
                        
                        svc1 = SVC(kernel=k1, class_weight={0: 1, 1: 4}, probability=True, C=C)
                        svc1.fit(X_tr, y_tr)
                        y_pred = svc1.predict(X_val)
                        prob_pos = svc1.predict_proba(X_val)[:, 1]
                        accuracies.append(accuracy_score(y_val, y_pred))
                        brier_scores.append(brier_score_loss(y_val, prob_pos))
                    
                    # Average accuracy across folds
                    mean_accuracy = np.mean(accuracies)
                    mean_brier = np.mean(brier_scores)
                    
                    # Update best model if current is better
                    if mean_accuracy > best_accuracy:
                        best_accuracy = mean_accuracy
                        best_params = {'gamma': γ, 'alpha': α, 'C': C}
                        best_model = svc1  
                        k1_best = Kernel1Full(
                                    continuous_vars=cont_in_sub,
                                    alpha=α, rbf_gamma=γ, cat_gamma=γ
                                  ).fit(X_sub, y_sub)
                        best_model = SVC(kernel=k1_best, class_weight={0:1, 1:4}, probability=True, C=C)

        best_model.fit(X_sub, y_sub)

        print(f"[seed {seed:03d}] best acc={best_accuracy:.4f}  params={best_params}")

        return best_model, best_accuracy, feat_idx, mean_brier
    # ------------------------------------------------------------------ #

    print(f"⇢ Training {no_svms} bagged SVMs …")
    results = []
    for s in range(no_svms):
        results.append(fit_one(s))

    models, accuracies, feature_idx, briers = zip(*results)
    ensemble = {
        "models":      list(models),
        "accuracies":  np.array(accuracies),
        "feature_idx": list(feature_idx),
        "briers": np.array(briers)
    }

    # ---- final summary ---------------------------------------------- #
    overall_best = np.max(ensemble["accuracies"])
    overall_brier = np.min(ensemble["briers"])
    print(f"✓ Finished all models — overall best CV accuracy = {overall_best:.4f}")
    print(f"✓ Finished all models — overall best CV brier = {overall_brier:.4f}")

    return ensemble




def predict_subBagSVM(ensemble, X_new):
    """
    Predict via majority vote from a subBagSVM ensemble.

    Parameters
    ----------
    ensemble : dict, The output of subBagSVM_k1 (with keys 'models' and 'feature_idx').
    X_new : pandas DataFrame or array-like, shape (n_samples_new, n_features)
        New feature matrix.

    Returns
    -------
    predictions : array, shape (n_samples_new,)
        Predicted labels by majority vote.
    """
    n_models = len(ensemble['models'])
    n_samples_new = X_new.shape[0]
    
    # collect each model's predictions
    votes = np.empty((n_samples_new, n_models), dtype=object)
    
    for i, (model, feat_idx) in enumerate(zip(ensemble['models'], ensemble['feature_idx'])):
        if hasattr(X_new, 'iloc'):  # pandas DataFrame
            columns = X_new.columns.tolist()
            feat_names = [columns[j] for j in feat_idx]
            X_sub = X_new[feat_names]
        else:  # numpy array
            X_sub = X_new[:, feat_idx]
        
        votes[:, i] = model.predict(X_sub)

    # majority vote
    def majority_label(row_votes):
        vals, counts = np.unique(row_votes, return_counts=True)
        return vals[np.argmax(counts)]

    preds = np.apply_along_axis(majority_label, axis=1, arr=votes)
    return preds


def subBagSVM_k1(X, y, m, cont_cols, no_svms=100, C_values=None, replace=True, n_jobs=1, random_state=None):

    rng = np.random.RandomState(random_state)
    n_samples, n_features = X.shape

    # Method fit one SVM with bootstrap + feature subspace
    def fit_one(seed):
        local_rng = np.random.RandomState(seed)
        idx_samples = local_rng.choice(n_samples, size=n_samples, replace=replace)
        feat_idx = local_rng.choice(n_features, size=m, replace=False)
        feat_names  = X.columns[feat_idx].tolist()
        X_sub = X.iloc[idx_samples, feat_idx]

        y_sub = y.iloc[idx_samples]

        cont_in_sub = [c for c in cont_cols if c in feat_names]


        gammas =  [0.5, 1.0, 2.0, 4.0] 
        alphas = [0.3, 0.5, 0.7, 1.0, 1.5]
        Cs = [0.1, 1, 10]

        # Initialize variables to track the best model
        best_accuracy = 0
        best_params = {}
        best_model = None

        # Add cross-validation (3-fold)
        kf = KFold(n_splits=3, shuffle=True, random_state=42)

        for γ in gammas:
            # print(f"Testing gamma={γ}")
            for α in alphas:
                for C in Cs:
                    accuracies = []
                    brier_scores = []

                    
                    # Cross-validation loop
                    for train_idx, val_idx in kf.split(X_sub):
                        X_tr, X_val = X_sub.iloc[train_idx], X_sub.iloc[val_idx]
                        y_tr, y_val = y_sub.iloc[train_idx], y_sub.iloc[val_idx]
                        
                        # Train kernel and SVC
                        k1 = Kernel1Full(
                            continuous_vars=cont_in_sub,
                            alpha=α,
                            rbf_gamma=γ,
                            cat_gamma=γ
                        ).fit(X_tr, y_tr)
                        
                        svc1 = SVC(kernel=k1, class_weight={0: 1, 1: 4}, probability=True, C=C)
                        svc1.fit(X_tr, y_tr)
                        y_pred = svc1.predict(X_val)
                        prob_pos = svc1.predict_proba(X_val)[:, 1]
                        accuracies.append(accuracy_score(y_val, y_pred))
                        brier_scores.append(brier_score_loss(y_val, prob_pos))
                    
                    # Average accuracy across folds
                    mean_accuracy = np.mean(accuracies)
                    mean_brier = np.mean(brier_scores)
                    
                    # Update best model if current is better
                    if mean_accuracy > best_accuracy:
                        best_accuracy = mean_accuracy
                        best_params = {'gamma': γ, 'alpha': α, 'C': C}
                        best_model = svc1  
                        k1_best = Kernel1Full(
                                    continuous_vars=cont_in_sub,
                                    alpha=α, rbf_gamma=γ, cat_gamma=γ
                                  ).fit(X_sub, y_sub)
                        best_model = SVC(kernel=k1_best, class_weight={0:1, 1:4}, probability=True, C=C)

        best_model.fit(X_sub, y_sub)

        print(f"[seed {seed:03d}] best acc={best_accuracy:.4f}  params={best_params}")

        return best_model, best_accuracy, feat_idx, mean_brier

    print(f"⇢ Training {no_svms} bagged SVMs …")
    results = []
    for s in range(no_svms):
        results.append(fit_one(s))

    models, accuracies, feature_idx, briers = zip(*results)
    ensemble = {
        "models":      list(models),
        "accuracies":  np.array(accuracies),
        "feature_idx": list(feature_idx),
        "briers": np.array(briers)
    }

    overall_best = np.max(ensemble["accuracies"])
    overall_brier = np.min(ensemble["briers"])
    print(f"✓ Finished all models — overall best CV accuracy = {overall_best:.4f}")
    print(f"✓ Finished all models — overall best CV brier = {overall_brier:.4f}")

    return ensemble


def predict_subBagSVM(ensemble, X_new):
    n_models = len(ensemble['models'])
    n_samples_new = X_new.shape[0]
    
    # collect each model's predictions
    votes = np.empty((n_samples_new, n_models), dtype=object)
    
    for i, (model, feat_idx) in enumerate(zip(ensemble['models'], ensemble['feature_idx'])):
        if hasattr(X_new, 'iloc'):  
            columns = X_new.columns.tolist()
            feat_names = [columns[j] for j in feat_idx]
            X_sub = X_new[feat_names]
        else: 
            X_sub = X_new[:, feat_idx]
        
        votes[:, i] = model.predict(X_sub)

    # majority vote
    def majority_label(row_votes):
        vals, counts = np.unique(row_votes, return_counts=True)
        return vals[np.argmax(counts)]

    preds = np.apply_along_axis(majority_label, axis=1, arr=votes)
    return preds

In [ ]:
# uncomment each section according to dataset being used
# # 1. dmean dataset
# df = pd.read_csv("dmean_df.csv")
# y = df["ExtractionFlag"]
# X = df.drop(columns=["ExtractionFlag"])

# # 2. dmax dataset
# # df = pd.read_csv("dmax_df.csv")
# # y = df["ExtractionFlag"]
# # X = df.drop(columns=["ExtractionFlag"])

# cont_cols = ["TotalDose"]   
# cat_cols = X.drop(columns=['TotalDose']).columns.tolist()

# #  this splitting procedure is only for extraction datasets
# unique_patients = df['PatientID'].unique()

# train_patients, test_patients = train_test_split(
#     unique_patients, 
#     test_size=0.2, 
#     random_state=42
# )

# train_mask = df['PatientID'].isin(train_patients)
# test_mask = df['PatientID'].isin(test_patients)

# X_train = X[train_mask]
# X_test = X[test_mask]
# y_train = y[train_mask]
# y_test = y[test_mask]

# X_train = X_train.drop(columns=["PatientID"])
# X_test = X_test.drop(columns=["PatientID"])

# print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

# df 3: 
congressional_voting_records = fetch_ucirepo(id=105) 
X = congressional_voting_records.data.features 
y = congressional_voting_records.data.targets 
y = y.iloc[:, 0] 
y = y.map({'republican': 0, 'democrat': 1})
cat_cols = X.columns.tolist()   
cont_cols = []  # No continuous columns in this dataset

# df 4: MONK dataset
# monk_s_problems = fetch_ucirepo(id=70) 
# X = monk_s_problems.data.features 
# y = monk_s_problems.data.targets 
# y = y.map({'republican': 0, 'democrat': 1})
# cat_cols = X.columns.tolist()   
# cont_cols = []  # No continuous columns in this dataset

# this split is used for the UCI datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

Training data: (1431, 12)
Testing data:  (358, 12)


In [ ]:
m_values = [4,6,8,10,12] # extraction datasets
m_values = [3,4,6] 
m_values = [6,9,12,16] 


no_svms = 50
C_values = [0.1, 1, 10] 
random_state = 42 
cont_cols = ["TotalDose"] 

# Store results 
results_k1_rsvm = [] 
results_k1_rsm = [] 

for m in m_values: 
    print(f"Training m={m} ...") 
    # ----- k1 RSVM ----- 
    ens_lin_rsvm = subBagSVM_k1(X_train,
                                y_train, 
                                m=m, 
                                cont_cols=["TotalDose"],
                                no_svms=no_svms, 
                                replace=True, 
                                n_jobs=-1, 
                                random_state=random_state) 
    accs_k1_rsvm = [] 
    for i in range(1, no_svms + 1): 
        sub_ens = {'models': ens_lin_rsvm['models'][:i], 'feature_idx': ens_lin_rsvm['feature_idx'][:i]} 
        pred = predict_subBagSVM(sub_ens, X_test.values) 
        accs_k1_rsvm.append(accuracy_score(y_test, pred)) 
    results_k1_rsvm.append(accs_k1_rsvm) 
    print("done training k1 RSVM")

accs_k1_rsm = []
brier_k1_rsm = []

for m in m_values: 
    # ----- k1 RSM ----- 

    ens_k1_rsm = subBagSVM_k1(X_train, 
                          y_train, 
                          m=m, 
                          cont_cols=["TotalDose"],
                          no_svms=no_svms,
                          replace=False, 
                          n_jobs=-1, 
                          random_state=random_state)
    
    print(f"Training m={m} ...") 
    accs_k1_rsm = [] 
    for i in range(1, no_svms + 1):
        sub_ens = {'models': ens_k1_rsm['models'][:i], 'feature_idx': ens_k1_rsm['feature_idx'][:i]}
        pred = predict_subBagSVM(sub_ens, X_test.values)
        accs_k1_rsm.append(accuracy_score(y_test, pred))
    results_k1_rsm.append(accs_k1_rsm)
print("done training k1 RSM")

colors = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(8, 6))

for i, m in enumerate(m_values):
    clr = colors[i % len(colors)]
    x = range(1, no_svms + 1)

    ax.plot(x, results_k1_rsvm[i], color=clr, linestyle='-',  label=f'm={m}')
    ax.plot(x, results_k1_rsm[i],  color=clr, linestyle='--')           


color_handles = [
    mlines.Line2D([], [], color=colors[i % len(colors)],
                  linestyle='-', linewidth=2, label=str(m))
    for i, m in enumerate(m_values)
]

style_handles = [
    mlines.Line2D([], [], color='k', linestyle='-',  linewidth=2, label='RSVM'),
    mlines.Line2D([], [], color='k', linestyle='--', linewidth=2, label='RSM'),
]

handles = color_handles + style_handles

# Final plot 
ax.set_title("K1 Kernel")
ax.set_xlabel("Number of SVMs")
ax.set_ylabel("Test Accuracy")

# place legend **below** the axes
ax.legend(handles=handles,
          loc='lower center',
          bbox_to_anchor=(0.5, -0.25),
          ncol= max(len(m_values), 3),     
          frameon=True)

fig.tight_layout()   
plt.show()